# Parietal 區 11 分類器 p-value 統計檢定
- **Friedman test** 整體差異
- **McNemar's test** (paired, session-level n=168) vs SVM-RBF
- **Wilcoxon signed-rank** (paired, subject-level n=56) vs SVM-RBF
- **Holm-Bonferroni** 校正多重比較

In [1]:
import os, time
import numpy as np
import h5py
import pandas as pd

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from scipy.stats import friedmanchisquare, wilcoxon, chi2 as chi2_dist

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
try:
    from lightgbm import LGBMClassifier
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
print(f'XGBoost={HAS_XGB}, LightGBM={HAS_LGB}')

XGBoost=True, LightGBM=True


In [2]:
# Load PLV + Band Energy data
PARIETAL = [13, 18, 14, 19, 12, 11, 22, 10, 21]
BAND_NAMES = ['delta','theta','alpha','beta','gamma']
CLASS_NAMES = ['amateur','master','prof']
BASE_PLV = {g: rf'D:\Tseng\圍棋\{g}\PLV_Results_2026' for g in CLASS_NAMES}
BASE_BE  = {g: rf'D:\Tseng\圍棋\{g}\band_energy_batch_outputs_2026' for g in CLASS_NAMES}

def get_id(fn): return fn.split('_ses-')[0]
def scan(folder): return {get_id(f): os.path.join(folder, f)
                          for f in sorted(os.listdir(folder)) if f.endswith('.mat')}
def load_plv(fp):
    with h5py.File(fp, 'r') as f:
        rm = f['result_mats']
        return np.array([np.array(rm[b]) for b in BAND_NAMES])
def load_be(fp):
    with h5py.File(fp, 'r') as f:
        return np.array(f['band_energy']).T
def load_grp(loader, m1, m2, m3, subs):
    return tuple(np.array([loader(m[s]) for s in subs]) for m in (m1, m2, m3))

t0 = time.time()
plv_maps, be_maps, complete = {}, {}, {}
for g in CLASS_NAMES:
    pm = {s: os.path.join(BASE_PLV[g], s) for s in ['ss01','ss02','ss03']}
    bm = {s: os.path.join(BASE_BE[g],  s) for s in ['ss01','ss02','ss03']}
    plv_maps[g] = (scan(pm['ss01']), scan(pm['ss02']), scan(pm['ss03']))
    be_maps[g]  = (scan(bm['ss01']), scan(bm['ss02']), scan(bm['ss03']))
    complete[g] = sorted(set(plv_maps[g][0]) & set(plv_maps[g][1]) & set(plv_maps[g][2]))

X_plv_g = {g: load_grp(load_plv, *plv_maps[g], complete[g]) for g in CLASS_NAMES}
X_be_g  = {g: load_grp(load_be,  *be_maps[g],  complete[g]) for g in CLASS_NAMES}
n_a, n_m, n_p = len(complete['amateur']), len(complete['master']), len(complete['prof'])

X_plv = np.vstack([x for g in CLASS_NAMES for x in X_plv_g[g]])
X_be  = np.vstack([x for g in CLASS_NAMES for x in X_be_g[g]])
y_all = np.concatenate([np.zeros(n_a*3), np.ones(n_m*3), np.full(n_p*3, 2)])
subject_ids = complete['amateur'] + complete['master'] + complete['prof']
subject_split = np.concatenate([np.repeat(complete[g], 3) for g in CLASS_NAMES])
print(f'Loaded {len(subject_ids)} subjects, {len(y_all)} sessions in {time.time()-t0:.1f}s')

Loaded 56 subjects, 168 sessions in 5.6s


In [3]:
# Extract Parietal features (PLV + BE)
ch = PARIETAL
Xm = X_plv.mean(axis=2)[:, :, np.ix_(ch, ch)[0], np.ix_(ch, ch)[1]]
tri = np.triu_indices(len(ch), k=1)
F_plv = np.hstack([Xm[:, b, tri[0], tri[1]] for b in range(5)])
F_be  = np.log(X_be + 1e-8)[:, :, 1:-1, :].mean(axis=2)[:, ch, :].reshape(len(X_be), -1)
print(f'PLV features: {F_plv.shape}   BE features: {F_be.shape}')

PLV features: (168, 180)   BE features: (168, 45)


In [4]:
# Define classifiers (same configuration as ml_plv_be_compare.ipynb)
classifiers = {
    'SVM-RBF':      lambda: SVC(kernel='rbf', C=10, gamma='scale', class_weight='balanced',
                                probability=True, random_state=42),
    'LogReg':       lambda: LogisticRegression(max_iter=5000, C=1.0, class_weight='balanced', random_state=42),
    'LDA':          lambda: LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr'),
    'KNN-5':        lambda: KNeighborsClassifier(n_neighbors=5, weights='distance'),
    'GaussianNB':   lambda: GaussianNB(),
    'RandomForest': lambda: RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                                   random_state=42, n_jobs=-1),
    'ExtraTrees':   lambda: ExtraTreesClassifier(n_estimators=300, class_weight='balanced',
                                                 random_state=42, n_jobs=-1),
    'GradBoost':    lambda: GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                                       max_depth=3, random_state=42),
    'MLP':          lambda: MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=1000,
                                          early_stopping=True, random_state=42),
}
if HAS_XGB:
    classifiers['XGBoost'] = lambda: XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                                                   tree_method='hist', eval_metric='mlogloss',
                                                   random_state=42, verbosity=0)
if HAS_LGB:
    classifiers['LightGBM'] = lambda: LGBMClassifier(n_estimators=300, learning_rate=0.05,
                                                     num_leaves=31, random_state=42, verbosity=-1)
clf_names = list(classifiers.keys())
print(f'{len(clf_names)} classifiers: {clf_names}')

11 classifiers: ['SVM-RBF', 'LogReg', 'LDA', 'KNN-5', 'GaussianNB', 'RandomForest', 'ExtraTrees', 'GradBoost', 'MLP', 'XGBoost', 'LightGBM']


In [5]:
# Run LOSO for each classifier on Parietal (a few minutes)
n_clf = len(clf_names)
acc_matrix = np.zeros((len(subject_ids), n_clf))    # per-subject accuracy
correct_matrix = np.zeros((168, n_clf), dtype=int)  # per-session correctness
overall_acc = {}

for j, name in enumerate(clf_names):
    t0 = time.time()
    factory = classifiers[name]
    all_p, all_t = [], []
    for i, test_sub in enumerate(subject_ids):
        m = subject_split == test_sub
        y_tr = y_all[~m].astype(int); y_te = y_all[m].astype(int)
        sc1 = StandardScaler(); p_tr = sc1.fit_transform(F_plv[~m]); p_te = sc1.transform(F_plv[m])
        pca1 = PCA(n_components=0.95); p_tr = pca1.fit_transform(p_tr); p_te = pca1.transform(p_te)
        sc2 = StandardScaler(); b_tr = sc2.fit_transform(F_be[~m]); b_te = sc2.transform(F_be[m])
        pca2 = PCA(n_components=0.95); b_tr = pca2.fit_transform(b_tr); b_te = pca2.transform(b_te)
        Xtr = np.hstack([p_tr, b_tr]); Xte = np.hstack([p_te, b_te])
        sc3 = StandardScaler(); Xtr = sc3.fit_transform(Xtr); Xte = sc3.transform(Xte)
        clf = factory(); clf.fit(Xtr, y_tr)
        preds = clf.predict(Xte)
        acc_matrix[i, j] = accuracy_score(y_te, preds)
        all_p.extend(preds); all_t.extend(y_te)
    all_p, all_t = np.array(all_p), np.array(all_t)
    correct_matrix[:, j] = (all_p == all_t).astype(int)
    overall_acc[name] = accuracy_score(all_t, all_p)
    print(f'  [{name:<13s}] Acc={overall_acc[name]:.4f}  ({time.time()-t0:.1f}s)')

  [SVM-RBF      ] Acc=0.8393  (1.9s)
  [LogReg       ] Acc=0.5833  (1.5s)
  [LDA          ] Acc=0.5774  (1.3s)
  [KNN-5        ] Acc=0.7619  (3.1s)
  [GaussianNB   ] Acc=0.6310  (1.1s)
  [RandomForest ] Acc=0.7381  (26.4s)
  [ExtraTrees   ] Acc=0.8095  (22.2s)
  [GradBoost    ] Acc=0.7202  (68.0s)
  [MLP          ] Acc=0.5000  (2.6s)
  [XGBoost      ] Acc=0.7024  (24.8s)


c:\Users\user\anaconda3\envs\game_eeg\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\user\anaconda3\envs\game_eeg\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\user\anaconda3\envs\game_eeg\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\user\anaconda3\envs\game_eeg\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\user\anaconda3\envs\game_eeg\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with f

  [LightGBM     ] Acc=0.7321  (7.2s)


c:\Users\user\anaconda3\envs\game_eeg\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\user\anaconda3\envs\game_eeg\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [6]:
# (1) Friedman test — overall difference among classifiers
chi2_f, p_friedman = friedmanchisquare(*[acc_matrix[:, j] for j in range(n_clf)])
print('=' * 70)
print(f'Friedman test on Parietal ({n_clf} classifiers x {len(subject_ids)} subjects)')
print('=' * 70)
print(f'  chi2 = {chi2_f:.3f}    p = {p_friedman:.3e}')
print('  -> ' + ('overall significant difference (p<0.05); proceed to post-hoc'
                 if p_friedman < 0.05 else 'no overall significant difference'))

Friedman test on Parietal (11 classifiers x 56 subjects)
  chi2 = 98.810    p = 9.427e-17
  -> overall significant difference (p<0.05); proceed to post-hoc


In [7]:
# (2) Pairwise vs SVM-RBF: McNemar (session-level) + Wilcoxon (subject-level) + Holm correction
def mcnemar_test(b, c):
    n = b + c
    if n == 0: return 0.0, 1.0
    chi2 = (abs(b - c) - 1) ** 2 / n
    return chi2, float(1 - chi2_dist.cdf(chi2, df=1))

def holm(pvals):
    pvals = np.asarray(pvals, float); n = len(pvals); order = np.argsort(pvals)
    adj = np.empty(n); prev = 0.0
    for i, idx in enumerate(order):
        v = min(1.0, pvals[idx] * (n - i)); prev = max(prev, v); adj[idx] = prev
    return adj

def stars(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

BASE = 'SVM-RBF'; bidx = clf_names.index(BASE)
rows = []
for j, clf in enumerate(clf_names):
    if clf == BASE: continue
    a = correct_matrix[:, bidx]; b = correct_matrix[:, j]
    b_only = int(((a==1)&(b==0)).sum()); c_only = int(((a==0)&(b==1)).sum())
    _, p_mc = mcnemar_test(b_only, c_only)
    diff = acc_matrix[:, bidx] - acc_matrix[:, j]
    if np.all(diff == 0):
        p_wil = 1.0
    else:
        try: _, p_wil = wilcoxon(diff, alternative='two-sided')
        except ValueError: p_wil = 1.0
    rows.append({'classifier': clf, 'acc': overall_acc[clf],
                 'acc_diff': overall_acc[BASE] - overall_acc[clf],
                 'svm_better': b_only, 'other_better': c_only,
                 'mcnemar_p': p_mc, 'wilcoxon_p': p_wil})

mc_h = holm([r['mcnemar_p'] for r in rows])
wi_h = holm([r['wilcoxon_p'] for r in rows])
for r, mh, wh in zip(rows, mc_h, wi_h):
    r['mcnemar_p_holm'] = mh; r['wilcoxon_p_holm'] = wh

print('=' * 110)
print(f'Pairwise vs {BASE} on Parietal (sorted by McNemar p; Holm across {len(rows)} pairs)')
print('=' * 110)
print(f'{"vs SVM-RBF":<14s} {"Acc":>6s} {"dAcc":>7s} {"SVM>":>5s} {"Other>":>7s} '
      f'{"McNemar p":>11s} {"Holm":>10s} {"sig":>4s} '
      f'{"Wilcox p":>11s} {"Holm":>10s} {"sig":>4s}')
print('-' * 110)
for r in sorted(rows, key=lambda x: x['mcnemar_p']):
    print(f'{r["classifier"]:<14s} {r["acc"]:6.3f} {r["acc_diff"]:+7.3f} '
          f'{r["svm_better"]:5d} {r["other_better"]:7d} '
          f'{r["mcnemar_p"]:11.3e} {r["mcnemar_p_holm"]:10.3e} {stars(r["mcnemar_p_holm"]):>4s} '
          f'{r["wilcoxon_p"]:11.3e} {r["wilcoxon_p_holm"]:10.3e} {stars(r["wilcoxon_p_holm"]):>4s}')

print()
print('stars (Holm-corrected):  *** p<0.001   ** p<0.01   * p<0.05   ns = not significant')
print('SVM>: # sessions SVM correct & other wrong    Other>: # sessions SVM wrong & other correct')

pd.DataFrame(rows).set_index('classifier').to_csv(
    r'D:\Tseng\game_eeg\ML_PLV_BE\ml_plv_be_pvalues_vs_svm.csv')
print('\nSaved: ml_plv_be_pvalues_vs_svm.csv')

Pairwise vs SVM-RBF on Parietal (sorted by McNemar p; Holm across 10 pairs)
vs SVM-RBF        Acc    dAcc  SVM>  Other>   McNemar p       Holm  sig    Wilcox p       Holm  sig
--------------------------------------------------------------------------------------------------------------
MLP             0.500  +0.339    61       4   3.759e-12  3.759e-11  ***   8.975e-07  7.474e-06  ***
LDA             0.577  +0.262    52       8   2.836e-08  2.552e-07  ***   8.305e-07  7.474e-06  ***
LogReg          0.583  +0.256    52       9   7.551e-08  6.041e-07  ***   2.560e-07  2.560e-06  ***
GaussianNB      0.631  +0.208    44       9   3.008e-06  2.106e-05  ***   3.368e-05  2.358e-04  ***
XGBoost         0.702  +0.137    33      10   7.937e-04  4.762e-03   **   3.850e-03  2.310e-02    *
LightGBM        0.732  +0.107    25       7   2.654e-03  1.327e-02    *   9.690e-03  4.845e-02    *
GradBoost       0.720  +0.119    30      10   2.663e-03  1.327e-02    *   1.442e-02  5.767e-02   ns
RandomForest 